In [49]:
import sys # importem mòdul sys que permet gestionar sistema i rutes de fitxers.
import os # importem mòdul os que permet interactuar S.O, com ara gestionar rutes de fitxers.
from datetime import date

# Afegim directori src a path per importar els mòduls del projecte.
sys.path.append(os.path.abspath("../src"))

from domain import AppCredential
from domain.config import DB_URL, ENVIRONMENT
from domain.unit_of_work import UnitOfWorkFactory
from domain.models import Category, Employee, Menu, Dish, Sale, Allergen

# Inicialitzem UnitOfWorkFactory amb URL de la bd's.
uow_factory = UnitOfWorkFactory(DB_URL)
print(f"UnitOfWorkFactory correctly initialized. Environment: {ENVIRONMENT}")
print(f"URL: {DB_URL}")

UnitOfWorkFactory correctly initialized. Environment: test
URL: sqlite:///:memory:


In [50]:
# Creem les taules des de zero per a les proves en memòria si Environment és 'test'
if ENVIRONMENT == "test":
    from domain.db import Base
    Base.metadata.drop_all(uow_factory.engine)
    Base.metadata.create_all(uow_factory.engine)
    print("Tables created in memory correctly.")

Tables created in memory correctly.


## 1. CategoryRepository

In [51]:
# add() - Creació de categories
print("Category add():")
with uow_factory.create() as uow:
    cat1 = Category(category_name="Arrossos")
    cat2 = Category(category_name="Carns")
    cat3 = Category(category_name="Infusions")
    uow.categories.add(cat1)
    uow.categories.add(cat2)
    uow.categories.add(cat3)

print(f"category creada - ID: {cat1.id}, Nom: {cat1.category_name}")
print(f"category creada - ID: {cat2.id}, Nom: {cat2.category_name}")
print(f"category creada - ID: {cat3.id}, Nom: {cat3.category_name}")

Category add():
category creada - ID: 1, Nom: Arrossos
category creada - ID: 2, Nom: Carns
category creada - ID: 3, Nom: Infusions


In [52]:
# Prova de validació - Categoria duplicada
print("Category validation test:")
with uow_factory.create() as uow:
    try:
        duplicate_cat = Category(category_name="Arrossos")
        uow.categories.add(duplicate_cat)
        uow.commit()
        print("ERROR: Duplicated category added without raising ValueError.")
    except ValueError as e:
        print(f"Correct validation! Message: {e}")

Category validation test:
Correct validation! Message: Category 'Arrossos' already exists.


In [53]:
# get() - Llegir per ID
print("Category get():")
with uow_factory.create() as uow:
    category = uow.categories.get(cat1.id)

if category is not None:
    print(f"Category found: ID={category.id}, Name={category.category_name}")
else:
    print("  Category not found.")

Category get():
Category found: ID=1, Name=Arrossos


In [54]:
# get_all() - Llegir totes les categories
print("Category get_all():")
with uow_factory.create() as uow:
    all_categories = uow.categories.get_all()

print(f"Total categories in the DB: {len(all_categories)}")
for cat in all_categories:
    print(f"  - {cat}")

Category get_all():
Total categories in the DB: 3
  - Arrossos
  - Carns
  - Infusions


In [55]:
# update() - Modificar una categoria
print("Category update():")
with uow_factory.create() as uow:
    category_to_update = uow.categories.get(cat1.id)
    category_to_update.category_name = "Arrossos"
    uow.categories.update(category_to_update)

with uow_factory.create() as uow:
    updated_category = uow.categories.get(cat1.id)
print(f"Category updated: {updated_category.category_name}")

Category update():
Category updated: Arrossos


In [56]:
# update() - Validació de nom duplicat
print("Category update():")
with uow_factory.create() as uow:
    try:
        category_to_update = uow.categories.get(cat1.id)
        category_to_update.category_name = "Carns"  # nom que ja existeix a cat2
        uow.categories.update(category_to_update)
        print("ERROR: Duplicate name was allowed!")
    except ValueError as e:
        print(f"Validation correct! Message: {e}")

Category update():
Validation correct! Message: Category name 'Carns' is already taken.


In [57]:
# get_by_name() - Cerca per nom
print("Category get_by_name():")
with uow_factory.create() as uow:
    result = uow.categories.get_by_name("Carns")

print(f"Category found by name: {result}")
if result is None:
    print("  No category found with this name.")

Category get_by_name():
Category found by name: Carns


In [58]:
# delete() - Eliminar una categoria
print("Category delete():")
with uow_factory.create() as uow:
    deleted = uow.categories.delete(cat3.id)

if deleted == None:
    print(f"Deletion completed.")

with uow_factory.create() as uow:
    check = uow.categories.get(cat3.id)
print(f"Deleted category (should be None): {check}")

Category delete():
Deletion completed.
Deleted category (should be None): None


## 2. AllergenRepository

In [59]:
# add() - Creació d'al·lèrgens
print("Allergen add():")
with uow_factory.create() as uow:
    all1 = Allergen(allergen_name="Mostassa")
    all2 = Allergen(allergen_name="Api")
    all3 = Allergen(allergen_name="Llúpol")
    uow.allergens.add(all1)
    uow.allergens.add(all2)
    uow.allergens.add(all3)

print(f"Al·lèrgens creats - IDs: {all1.id}, {all2.id}, {all3.id}")

Allergen add():
Al·lèrgens creats - IDs: 1, 2, 3


In [60]:
# get(), get_all(), get_by_name()
print("AllergenRepository get() / get_all() / get_by_name():")
with uow_factory.create() as uow:
    allergen = uow.allergens.get(all1.id)
    all_allergens = uow.allergens.get_all()
    by_name = uow.allergens.get_by_name("Api")

print(f"get(): {allergen}")
print(f"get_all() - total: {len(all_allergens)}")
print(f"get_by_name('Api'): {by_name}")

AllergenRepository get() / get_all() / get_by_name():
get(): Mostassa
get_all() - total: 3
get_by_name('Api'): Api


In [61]:
# update()
print("AllergenRepository update() and delete():")
with uow_factory.create() as uow:
    a = uow.allergens.get(all3.id)
    a.allergen_name = "Diòxid de sofre"
    uow.allergens.update(a)

with uow_factory.create() as uow:
    updated_allergen = uow.allergens.get(all3.id)
print(f"Updated allergen: {updated_allergen.allergen_name}")

#delete()
with uow_factory.create() as uow:
    uow.allergens.delete(all3.id)

with uow_factory.create() as uow:
    check = uow.allergens.get(all3.id)
print(f"Deleted allergen (should be None): {check}")


AllergenRepository update() and delete():
Updated allergen: Diòxid de sofre
Deleted allergen (should be None): None


## 3. DishRepository

In [62]:
# add() - Creació de plats
# Fem servir cat1 i cat2 que hem creat en aquest mateix notebook
print("Dish add():")
with uow_factory.create() as uow:
    dish1 = Dish(dish_name="Arròs negre", description="Arròs amb tinta de calamar", is_active=True, id_category=cat1.id)
    dish2 = Dish(dish_name="Entrecot de vedella", description="Vedella de 250g amb guarnició", is_active=True, id_category=cat2.id)
    dish3 = Dish(dish_name="Arròs a banda", description="Arròs de mar tradicional", is_active=False, id_category=cat1.id)
    uow.dishes.add(dish1)
    uow.dishes.add(dish2)
    uow.dishes.add(dish3)

print(f"Plats creats - IDs: {dish1.id}, {dish2.id}, {dish3.id}")

Dish add():
Plats creats - IDs: 1, 2, 3


In [63]:
# get() i get_all()
print("Dish get() / get_all():")
with uow_factory.create() as uow:
    dish = uow.dishes.get(dish1.id)
    all_dishes = uow.dishes.get_all()

print(f"get(): {dish}")
print(f"get_all() - total: {len(all_dishes)}")

Dish get() / get_all():
get(): Arròs negre
get_all() - total: 3


In [64]:
# get_active_dishes() - Cerca plats actius
print("Dish get_active_dishes():")
with uow_factory.create() as uow:
    active_dishes = uow.dishes.get_active_dishes()

print(f"Active dishes found: {len(active_dishes)}")
for d in active_dishes:
    print(f"  - {d.dish_name} (active: {d.is_active})")
print("Note: 'Arros a banda' should NOT appear because is_active=False")

Dish get_active_dishes():
Active dishes found: 2
  - Arròs negre (active: True)
  - Entrecot de vedella (active: True)
Note: 'Arros a banda' should NOT appear because is_active=False


In [65]:
# get_by_category() - Cerca per categoria
print("Dish get_by_category():")
with uow_factory.create() as uow:
    dishes_cat1 = uow.dishes.get_by_category(cat1.id)

print(f"Dishes in category '{cat1.category_name}' (ID {cat1.id}):")
for d in dishes_cat1:
    print(f"  - {d.dish_name}")

Dish get_by_category():
Dishes in category 'Arrossos' (ID 1):
  - Arròs negre
  - Arròs a banda


In [66]:
# get_by_name() - Cerca per nom
print("Dish get_by_name():")
with uow_factory.create() as uow:
    results = uow.dishes.get_by_name("Arròs")

print(f"Dishes found with 'Arros': {results}")

Dish get_by_name():
Dishes found with 'Arros': [Dish(id=1, name='Arròs negre'), Dish(id=3, name='Arròs a banda')]


In [67]:
# add_allergen_to_dish() - Afegir al·lèrgen a un plat
print("Dish add_allergen_to_dish():")
with uow_factory.create() as uow:
    allergen_mustard = uow.allergens.get(all1.id)
    result = uow.dishes.add_allergen_to_dish(dish1.id, allergen_mustard)

print(f"Allergen added to dish: {result}")

with uow_factory.create() as uow:
    d = uow.dishes.get(dish1.id)
    print(f"Allergens of '{d.dish_name}': {d.allergens}")

Dish add_allergen_to_dish():
Allergen added to dish: True
Allergens of 'Arròs negre': [Allergen(id=1, name='Mostassa')]


In [68]:
# update() - Activar un plat que estava inactiu
print("Dish update():")
with uow_factory.create() as uow:
    d = uow.dishes.get(dish3.id)
    d.is_active = True
    uow.dishes.update(d)

with uow_factory.create() as uow:
    updated_dish = uow.dishes.get(dish3.id)
print(f"Dish '{updated_dish.dish_name}' - is_active updated to: {updated_dish.is_active}")

Dish update():
Dish 'Arròs a banda' - is_active updated to: True


## 4. EmployeeRepository

In [69]:
# add() - Creació d'empleats
# DNIs completament diferents als del populate (que usa 11111111A..66666666F)
print("Employee add():")
with uow_factory.create() as uow:
    emp1 = Employee(
        name="Carles", surname="Bosch", national_id="77777777G", role="Cambrer",
        credentials=AppCredential(email="carles@test.cat", password_hash="hash_carles")
    )
    emp2 = Employee(
        name="Núria", surname="Camps", national_id="88888888H", role="Cuinera",
        credentials=AppCredential(email="nuria@test.cat", password_hash="hash_nuria")
    )
    emp3 = Employee(
        name="Ferran", surname="Domènech", national_id="99999999I", role="Cambrer"
    )
    uow.employees.add(emp1)
    uow.employees.add(emp2)
    uow.employees.add(emp3)

print(f"Empleats creats - IDs: {emp1.id}, {emp2.id}, {emp3.id}")

Employee add():
Empleats creats - IDs: 1, 2, 3


In [70]:
# Prova de validació - DNI duplicat
print("Employee validation test:")
with uow_factory.create() as uow:
    try:
        duplicate_emp = Employee(
            name="Impostor", surname="Gimenez", national_id="77777777G", role="Cambrer"
        )
        uow.employees.add(duplicate_emp)
        uow.commit()
        print("ERROR: Duplicate national ID was allowed!")
    except ValueError as e:
        print(f"Validation correct! Message: {e}")

Employee validation test:
Validation correct! Message: Employee with national ID '77777777G' already exists.


In [71]:
# get() - Llegir per ID
print("Employee get():")
with uow_factory.create() as uow:
    employee = uow.employees.get(emp1.id)

print(f"Employee found: {employee}")
if employee is not None:
    print(f"  Name: {employee.name}, Role: {employee.role}")

Employee get():
Employee found: 1: Bosch, Carles (Cambrer)
  Name: Carles, Role: Cambrer


In [72]:
# get_all() - Llegir tots els empleats
print("Employee get_all():")
with uow_factory.create() as uow:
    all_employees = uow.employees.get_all()

print(f"Total employees in DB: {len(all_employees)}")
for emp in all_employees:
    print(f"  - {emp}")

Employee get_all():
Total employees in DB: 3
  - 1: Bosch, Carles (Cambrer)
  - 2: Camps, Núria (Cuinera)
  - 3: Domènech, Ferran (Cambrer)


In [73]:
# update() - Modificar un empleat
print("Employee update():")
with uow_factory.create() as uow:
    employee_to_update = uow.employees.get(emp1.id)
    employee_to_update.role = "Encarregat"
    uow.employees.update(employee_to_update)

with uow_factory.create() as uow:
    updated_employee = uow.employees.get(emp1.id)
print(f"Employee '{updated_employee.name}' - role updated to: {updated_employee.role}")

Employee update():
Employee 'Carles' - role updated to: Encarregat


In [74]:
# update() - Validació de DNI duplicat
print("Employee update() validation:")
with uow_factory.create() as uow:
    try:
        new_dni = "88888888H"  # DNI que ja existeix a emp2
        if uow.employees.national_id_exists(new_dni):
            raise ValueError(f"Cannot update. Another employee already has the national ID: {new_dni}")
        
        employee_to_update = uow.employees.get(emp1.id)
        employee_to_update.national_id = new_dni
        uow.employees.update(employee_to_update)
        print("ERROR: Duplicate national ID was allowed!")
    except ValueError as e: # Si es detecta el DNI duplicat, es captura l'error i es mostra el missatge de validació.
        print(f"Validation correct! Message: {e}")

Employee update() validation:
Validation correct! Message: Cannot update. Another employee already has the national ID: 88888888H


In [75]:
# get_by_name() - Cerca per nom o cognom
print("Employee get_by_name():")
with uow_factory.create() as uow:
    results = uow.employees.get_by_name(name="Núria")

print(f"Employees found with name 'Núria': {results}")

Employee get_by_name():
Employees found with name 'Núria': [Employee(id=2, name='Núria', role='Cuinera')]


In [76]:
# get_by_role() - Cerca per rol
print("Employee get_by_role():")
with uow_factory.create() as uow:
    waiters = uow.employees.get_by_role("Cambrer")

print(f"Waiters found: {len(waiters)}")
for emp in waiters:
    print(f"  - {emp}")

Employee get_by_role():
Waiters found: 1
  - 3: Domènech, Ferran (Cambrer)


In [77]:
# national_id_exists() - Validació de DNI
print("Employee national_id_exists():")
with uow_factory.create() as uow:
    exists_yes = uow.employees.national_id_exists("77777777G")
    exists_no = uow.employees.national_id_exists("00000000Z")

    print(f"National ID '77777777G' exists: {exists_yes} (should be True)")
    print(f"National ID '00000000Z' exists: {exists_no} (should be False)")

Employee national_id_exists():
National ID '77777777G' exists: True (should be True)
National ID '00000000Z' exists: False (should be False)


In [78]:
# get_paginated() - Paginació
print("Employee get_paginated():")
with uow_factory.create() as uow:
    page1 = uow.employees.get_paginated(page=1, page_size=2)
    page2 = uow.employees.get_paginated(page=2, page_size=2)

    print(f"Page 1 (2 per page): {page1}")
    print(f"Page 2 (2 per page): {page2}")

Employee get_paginated():
Page 1 (2 per page): [Employee(id=1, name='Carles', role='Encarregat'), Employee(id=2, name='Núria', role='Cuinera')]
Page 2 (2 per page): [Employee(id=3, name='Ferran', role='Cambrer')]


In [79]:
# delete() - Eliminar un empleat
print("Employee delete():")
with uow_factory.create() as uow:
    deleted = uow.employees.delete(emp3.id)

if deleted == None:
    print(f"Deletion completed.")

with uow_factory.create() as uow:
    check = uow.employees.get(emp3.id)
print(f"Deleted employee (should be None): {check}")

Employee delete():
Deletion completed.
Deleted employee (should be None): None


## 5. MenuRepository

In [80]:
# add() - Creació de menús
print("MenuRepository add():")
with uow_factory.create() as uow:
    menu1 = Menu(menu_name="Menú Executiu", price=14.00)
    menu2 = Menu(menu_name="Menú Degustació", price=38.00)
    menu3 = Menu(menu_name="Menú Infantil", price=7.50)
    uow.menus.add(menu1)
    uow.menus.add(menu2)
    uow.menus.add(menu3)

print(f"Menús creats - IDs: {menu1.id}, {menu2.id}, {menu3.id}")

MenuRepository add():
Menús creats - IDs: 1, 2, 3


In [81]:
# get() i get_all()
print("Menu get() / get_all():")
with uow_factory.create() as uow:
    menu = uow.menus.get(menu1.id)
    all_menus = uow.menus.get_all()

print(f"get(): {menu}")
print(f"get_all() - total: {len(all_menus)}")

Menu get() / get_all():
get(): Menú Executiu (14.00€)
get_all() - total: 3


In [82]:
# add_dish_to_menu() - Afegir plats al menú
print("Menu add_dish_to_menu():")
with uow_factory.create() as uow:
    d1 = uow.dishes.get(dish1.id)
    d2 = uow.dishes.get(dish2.id)
    uow.menus.add_dish_to_menu(menu1.id, d1)
    uow.menus.add_dish_to_menu(menu1.id, d2)

with uow_factory.create() as uow:
    m = uow.menus.get(menu1.id)
    print(f"Plats del '{m.menu_name}': {m.dishes}")

Menu add_dish_to_menu():
Plats del 'Menú Executiu': [Dish(id=1, name='Arròs negre'), Dish(id=2, name='Entrecot de vedella')]


In [83]:
# remove_dish_from_menu() - Treure un plat del menú
print("Menu remove_dish_from_menu():")
with uow_factory.create() as uow:
    d2 = uow.dishes.get(dish2.id)
    result = uow.menus.remove_dish_from_menu(menu1.id, d2)

print(f"Dish removed from menu: {result}")
with uow_factory.create() as uow:
    m = uow.menus.get(menu1.id)
    print(f"Remaining dishes in menu: {m.dishes}")

Menu remove_dish_from_menu():
Dish removed from menu: True
Remaining dishes in menu: [Dish(id=1, name='Arròs negre')]


In [84]:
# get_by_name() - Cerca per nom
print("Menu get_by_name():")
with uow_factory.create() as uow:
    results = uow.menus.get_by_name("Degustació")

    print(f"Menus found with 'Degustacio': {results}")

Menu get_by_name():
Menus found with 'Degustacio': [Menu(id=2, name='Menú Degustació', price=38.00)]


In [85]:
# get_by_price_range() - Cerca per rang de preus
print("Menu get_by_price_range():")
with uow_factory.create() as uow:
    affordable = uow.menus.get_by_price_range(0, 15.00)

print(f"Menus up to 15€:")
for m in affordable:
    print(f"  - {m.menu_name}: {m.price}€")
print("Note: 'Menu Executiu' (14€) and 'Menu Infantil' (7.50€) should appear")

Menu get_by_price_range():
Menus up to 15€:
  - Menú Executiu: 14.00€
  - Menú Infantil: 7.50€
Note: 'Menu Executiu' (14€) and 'Menu Infantil' (7.50€) should appear


In [86]:
# update() - Modificar el preu d'un menú
print("MenuRepository update():")
with uow_factory.create() as uow:
    m = uow.menus.get(menu1.id)
    m.price = 15.00
    uow.menus.update(m)

with uow_factory.create() as uow:
    updated_menu = uow.menus.get(menu1.id)
print(f"Updated price of '{updated_menu.menu_name}': {updated_menu.price}€")

MenuRepository update():
Updated price of 'Menú Executiu': 15.00€


## 6. SaleRepository

In [87]:
# add() - Creació de vendes
# Fem servir emp1 i emp2 (creats en aquest notebook) i dates de maig 2026
# (el populate usa dates de març 2026, no hi ha conflicte)
print("Sale add():")
with uow_factory.create() as uow:
    sale1 = Sale(id_menu=menu1.id, id_employee=emp1.id, sale_date=date(2026, 5, 10), quantity_sold=3)
    sale2 = Sale(id_menu=menu2.id, id_employee=emp1.id, sale_date=date(2026, 5, 10), quantity_sold=1)
    sale3 = Sale(id_menu=menu1.id, id_employee=emp2.id, sale_date=date(2026, 5, 11), quantity_sold=5)
    sale4 = Sale(id_menu=menu3.id, id_employee=emp2.id, sale_date=date(2026, 5, 12), quantity_sold=2)
    uow.sales.add(sale1)
    uow.sales.add(sale2)
    uow.sales.add(sale3)
    uow.sales.add(sale4)

print("Vendes creades correctament.")

Sale add():
Vendes creades correctament.


In [88]:
# get() - La Sale té clau primària composta (tuple)
print("Sale get():")
with uow_factory.create() as uow:
    # La clau primària de Sale és (id_menu, id_employee, sale_date)
    sale = uow.sales.get((menu1.id, emp1.id, date(2026, 5, 10)))

print(f"Sale found: {sale}")
if sale is not None:
    print(f"  Quantity sold: {sale.quantity_sold}")

Sale get():
Sale found: Venda del 2026-05-10: Menú ID 1 venut per Empleat ID 1
  Quantity sold: 3


In [89]:
# get_all()
print("Sale get_all():")
with uow_factory.create() as uow:
    all_sales = uow.sales.get_all()

print(f"Total sales in DB: {len(all_sales)}")
for s in all_sales:
    print(f"  - {s}")

Sale get_all():
Total sales in DB: 4
  - Venda del 2026-05-10: Menú ID 1 venut per Empleat ID 1
  - Venda del 2026-05-10: Menú ID 2 venut per Empleat ID 1
  - Venda del 2026-05-11: Menú ID 1 venut per Empleat ID 2
  - Venda del 2026-05-12: Menú ID 3 venut per Empleat ID 2


In [90]:
# update() - Modificar la quantitat d'una venda
print("Sale update():")
with uow_factory.create() as uow:
    s = uow.sales.get((menu1.id, emp1.id, date(2026, 5, 10)))
    s.quantity_sold = 10
    uow.sales.update(s)

with uow_factory.create() as uow:
    updated_sale = uow.sales.get((menu1.id, emp1.id, date(2026, 5, 10)))
print(f"Updated quantity: {updated_sale.quantity_sold} (was 3 before)")

Sale update():
Updated quantity: 10 (was 3 before)


In [91]:
# get_sales_by_date() - Cerca per data
print("Sale get_sales_by_date():")
with uow_factory.create() as uow:
    sales_by_day = uow.sales.get_sales_by_date(date(2026, 5, 10))

print(f"Sales on 2026-05-10: {sales_by_day}")
print(f"  Total: {len(sales_by_day)} (should be 2: sale1 and sale2)")

Sale get_sales_by_date():
Sales on 2026-05-10: [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=2, emp=1, date=2026-05-10)]
  Total: 2 (should be 2: sale1 and sale2)


In [92]:
# get_sales_by_employee() - Cerca per empleat
print("Sale get_sales_by_employee():")
with uow_factory.create() as uow:
    sales_emp1 = uow.sales.get_sales_by_employee(emp1.id)

print(f"Sales by employee {emp1.id} ({emp1.name}): {sales_emp1}")
print(f"  Total: {len(sales_emp1)} (should be 2: sale1 and sale2)")

Sale get_sales_by_employee():
Sales by employee 1 (Carles): [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=2, emp=1, date=2026-05-10)]
  Total: 2 (should be 2: sale1 and sale2)


In [93]:
# get_sales_by_menu() - Cerca per menú
print("Sale get_sales_by_menu():")
with uow_factory.create() as uow:
    sales_menu1 = uow.sales.get_sales_by_menu(menu1.id)

print(f"Sales of menu {menu1.id} ('{menu1.menu_name}'): {sales_menu1}")
print(f"  Total: {len(sales_menu1)} (should be 2: sale1 and sale3)")

Sale get_sales_by_menu():
Sales of menu 1 ('Menú Executiu'): [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=1, emp=2, date=2026-05-11)]
  Total: 2 (should be 2: sale1 and sale3)


In [94]:
# get_sales_by_date_range() - Cerca per rang de dates
print("Sale get_sales_by_date_range():")
with uow_factory.create() as uow:
    sales_range = uow.sales.get_sales_by_date_range(date(2026, 5, 10), date(2026, 5, 11))
    sales_empty = uow.sales.get_sales_by_date_range(date(2026, 6, 1), date(2026, 6, 30))

print(f"Sales from May 10 to 11: {sales_range}")
print(f"  Total: {len(sales_range)} (should be 3: sale1, sale2 and sale3)")
print(f"Sales in June (should be empty): {sales_empty}")

Sale get_sales_by_date_range():
Sales from May 10 to 11: [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=2, emp=1, date=2026-05-10), Sale(menu=1, emp=2, date=2026-05-11)]
  Total: 3 (should be 3: sale1, sale2 and sale3)
Sales in June (should be empty): []


In [95]:
# get_paginated() - Paginació de vendes
print("Sale get_paginated():")
with uow_factory.create() as uow:
    page1 = uow.sales.get_paginated(page=1, page_size=2)
    page2 = uow.sales.get_paginated(page=2, page_size=2)

print(f"Page 1 (2 per page): {page1}")
print(f"Page 2 (2 per page): {page2}")

Sale get_paginated():
Page 1 (2 per page): [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=2, emp=1, date=2026-05-10)]
Page 2 (2 per page): [Sale(menu=1, emp=2, date=2026-05-11), Sale(menu=3, emp=2, date=2026-05-12)]


In [96]:
# delete() - Eliminar una venda
print("SaleRepository: delete()")
with uow_factory.create() as uow:
    deleted = uow.sales.delete((menu3.id, emp2.id, date(2026, 5, 12)))

print(f"Deletion completed: {deleted}")

with uow_factory.create() as uow:
    check = uow.sales.get((menu3.id, emp2.id, date(2026, 5, 12)))
print(f"Deleted sale (should be None): {check}")

SaleRepository: delete()
Deletion completed: None
Deleted sale (should be None): None
